In [0]:
from pyspark.sql import functions as F

In [0]:
silver_source_table = "retail_project.silver.sales_cleaned"
dim_customers_table = "retail_project.gold.dim_customers"
dim_products_table = "retail_project.gold.dim_products"
dim_payments_table = "retail_project.gold.dim_payment"
target_table = "retail_project.gold.fact_sales"

In [0]:
silver_sales = spark.table(silver_source_table).alias("sales")
dim_cust = spark.table(dim_customers_table).filter("is_current = True").dropDuplicates(["cust_id"]).alias("cust")
dim_prod = spark.table(dim_products_table).alias("prod")
dim_pay = spark.table(dim_payments_table).alias("pay")

In [0]:
fact_df = (
    silver_sales
    .join(dim_cust, F.col("sales.cust_id") == F.col("cust.cust_id"), "inner")
    .join(dim_prod, F.col("sales.product_name") == F.col("prod.product_name"), "left")
    .join(dim_pay, 
        (F.col("sales.payment_mode") == F.col("pay.payment_mode")) & 
        (F.col("sales.payment_status") == F.col("pay.payment_status")) & 
        (F.col("sales.order_status") == F.col("pay.order_status")) & 
        (F.col("sales.found_us_via") == F.col("pay.found_us_via")), "left")
)

In [0]:
final_fact = fact_df.select(
    F.col("sales.order_id"),
    F.col("cust.customer_key"),
    F.col("prod.product_key"),
    F.col("pay.payment_key"),
    F.date_format(F.col("sales.event_ts"), "yyyyMMdd").cast("int").alias("date_key"),
    F.col("sales.base_price"),
    F.col("sales.discount_amount"),
    F.col("sales.final_price"),
    F.col("sales.year"),
    F.col("sales.month")
)

In [0]:
(final_fact.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("year", "month")
 .saveAsTable("retail_project.gold.fact_sales"))